In [68]:
import pandas as pd
import ml

from pandas.tseries.offsets import MonthEnd

In [69]:
meta = pd.read_csv('FRED-MD_updated_appendix.csv', encoding='cp1252', index_col = 0)

In [70]:
df_raw = pd.read_csv('2025-06-MD.csv', index_col=0, skiprows=[1])

In [71]:
df_raw.index = pd.DatetimeIndex(df_raw.index, freq='MS')

In [72]:
meta[meta['description'].str.contains('CPI')]

,tcode,fred,description,gsi,gsi:description,group
id,,,,,,
113,6,CPIAUCSL,CPI : All Items,M_110157323,CPI-U: all,7
114,6,CPIAPPSL,CPI : Apparel,M_110157299,CPI-U: apparel,7
115,6,CPITRNSL,CPI : Transportation,M_110157302,CPI-U: transp,7
116,6,CPIMEDSL,CPI : Medical Care,M_110157304,CPI-U: medical,7
117,6,CUSR0000SAC,CPI : Commodities,M_110157314,CPI-U: comm.,7
118,6,CUSR0000SAD,CPI : Durables,M_110157315,CPI-U: dbles,7
119,6,CUSR0000SAS,CPI : Services,M_110157325,CPI-U: services,7
120,6,CPIULFSL,CPI : All Items Less Food,M_110157328,CPI-U: ex food,7
121,6,CUSR0000SA0L2,CPI : All items less shelter,M_110157329,CPI-U: ex shelter,7


In [73]:
transformer = ml.FredMdTransformer(meta, id_col='fred', tcode_col='tcode')

In [74]:
df = transformer.transform(df_raw)

In [75]:
df = df.fillna(method = 'ffill')
df = df.dropna(axis = 0)

C:\Users\BOK\AppData\Local\Temp\ipykernel_135968\3533991242.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')


### Random Forest 학습 및 예측

In [135]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pandas.tseries.offsets import MonthBegin
import numpy as np

In [87]:
tscv = TimeSeriesSplit(n_splits = 3, test_size = 1)

In [78]:
X = df.drop(columns=['CPIAUCSL'])
y = df['CPIAUCSL']

#### 예측시계에 맞춰 X, y 조정

In [ ]:
hor = 1 # 1개월 뒤 예측
y_shifted = y.shift(-hor) # 예측시점 조정

In [206]:
X2 = X.copy()
y2 = y_shifted.copy()

Xy = pd.concat([X2, y2], axis=1).dropna()
X2 = Xy.drop(columns = 'CPIAUCSL')
y2 = Xy['CPIAUCSL']

In [224]:
actual = []
pred = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X2), 1):
    print(f"Fold {fold}")
    tm = (y2.index[test_idx] + MonthBegin(hor)).strftime('%Y-%m') # target month
    print(f"Target month: {tm[0]}") # 예측시계별 Xy 조정으로 실제 예측시점은 1개월 뒤 

    X_train, X_test = X2.iloc[train_idx], X2.iloc[test_idx]
    y_train, y_test = y2.iloc[train_idx], y2.iloc[test_idx]
    
    model = RandomForestRegressor(n_estimators=1000, max_depth=3, max_features=0.5, random_state=42) # random_state=42
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    actual.append(np.array(y_test))
    pred.append(y_pred)

print("평균 MAE:", np.round(mean_absolute_error(actual, pred), 4))
print("평균 RMSE:", np.round(np.sqrt(mean_squared_error(actual, pred)), 4))

Fold 1
Target month: 2025-03
Fold 2
Target month: 2025-04
Fold 3
Target month: 2025-05
평균 MAE: 0.1217
평균 RMSE: 0.1371


#### Decision tree와 비교

In [211]:
from sklearn.tree import DecisionTreeRegressor

In [213]:
actual = []
pred = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X2), 1):
    print(f"Fold {fold}")
    tm = (y2.index[test_idx] + MonthBegin(hor)).strftime('%Y-%m') # target month
    print(f"Target month: {tm[0]}") # 예측시계별 Xy 조정으로 실제 예측시점은 1개월 뒤 

    X_train, X_test = X2.iloc[train_idx], X2.iloc[test_idx]
    y_train, y_test = y2.iloc[train_idx], y2.iloc[test_idx]
    
    model = DecisionTreeRegressor(max_depth=7, max_features=0.5, random_state=42) # random_state=42
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    actual.append(np.array(y_test))
    pred.append(y_pred)

print("평균 MAE:", np.round(mean_absolute_error(actual, pred), 4))
print("평균 RMSE:", np.round(np.sqrt(mean_squared_error(actual, pred)), 4))

Fold 1
Target month: 2025-03
Fold 2
Target month: 2025-04
Fold 3
Target month: 2025-05
평균 MAE: 1.0184
평균 RMSE: 1.0768


### 하이퍼 파라미터 튜닝

In [214]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

In [ ]:
# 파라미터 그리드 정의
param_grid = {
    'n_estimators': [1000],
    'max_depth': [3, 5, 7, 10],
    'max_features': [0.3, 0.5, 0.7, 1.0]
}

# MAE를 최소화하는 방향으로 튜닝
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

model = RandomForestRegressor(random_state=42)

# grid_search 인스턴스 생성
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=mae_scorer,
    cv=tscv,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X2, y2)

print("최적 하이퍼파라미터:", grid_search.best_params_)
print("최적 MAE:", round(-grid_search.best_score_, 4)) # -붙이는 이유는 최소화하는 방향으로 튜닝하기 때문

Fitting 3 folds for each of 16 candidates, totalling 48 fits
최적 하이퍼파라미터: {'max_depth': 3, 'max_features': 0.5, 'n_estimators': 1000}
최적 MAE: 0.1217


### 모형 추가하기

In [226]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [227]:
# 모델 정의
models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=1000, max_depth=3, max_features=0.5, random_state=42
    ),
    'Ridge': Pipeline([
        ('scale', StandardScaler()),
        ('ridge', Ridge(alpha=1.0, random_state=42))
    ]),
    'Lasso': Pipeline([
        ('scale', StandardScaler()),
        ('lasso', Lasso(alpha=0.1, random_state=42))
    ]),
    'ElasticNet': Pipeline([
        ('scale', StandardScaler()),
        ('enet', ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42))
    ]),
}

In [228]:
results = {name: {'actual': [], 'pred': []} for name in models} # 결과 저장용 딕셔너리

In [ ]:
# 예측시계
hor = 1

# Walk-forward
for fold, (train_idx, test_idx) in enumerate(tscv.split(X2), 1):
    print(f"Fold {fold}")
    tm = (y2.index[test_idx] + MonthBegin(hor)).strftime('%Y-%m')
    print(f" Target month: {tm[0]}")

    X_train, X_test = X2.iloc[train_idx], X2.iloc[test_idx]
    y_train, y_test = y2.iloc[train_idx], y2.iloc[test_idx]

    for name, model in models.items():
        # 모델 학습
        model.fit(X_train, y_train)
        # 예측
        y_pred = model.predict(X_test)
        # 결과 저장
        idx = y2.index[test_idx] + MonthBegin(hor)
        results[name]['actual'].append(pd.Series(y_test.values, index=idx))
        results[name]['pred'].append(pd.Series(y_pred, index=idx))

# 결과
for name in models:
    actual = pd.concat(results[name]['actual']).sort_index()
    pred   = pd.concat(results[name]['pred']).sort_index()
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    print(f"{name:12s} → MAE: {mae:.4f}, RMSE: {rmse:.4f}")